# Chapter 1 — Responses API Fundamentals · Homework (self-test)

**Goal:** check that you can reason about the Responses API contract the way an architect would, and implement the pieces that production code needs.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- **Estimated time:** 60–90 minutes
- **Estimated API cost:** $0 for Parts A–C; the optional live exercise costs well under $0.01 with `gpt-6-luna`
- **Prerequisites:** README read, practice notebook done

**How to use it**

1. Attempt everything **without peeking**. Solutions are at the very bottom, inside collapsed blocks.
2. **Part A:** 12 scenario questions; put your letters in the `answers` dict and run the grader.
3. **Part B:** 5 offline coding exercises (auto-checked) plus 1 optional live exercise (not scored).
4. **Part C:** one architecture scenario; compare your answer with the rubric.
5. **Scorecard:** overall score = (correct quiz answers + offline exercises passed) / 17. The pass mark is **72%**, the same bar as the Claude path (which mirrors the Claude exam's 720/1000).

In [ ]:
import hashlib
import os
from types import SimpleNamespace

import openai
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from openai.types.responses import Response

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
print("Model:", MODEL)

In [ ]:
# Needed only for the optional live exercise. Everything else works without a key.
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env (see 00-prerequisites)"
client = OpenAI()

### Shared helpers (given)

Run this cell. `PRICES` and `SUPPORTED_EFFORT` come from the model pages (2026-10-03). `mock_response()` builds real SDK `Response` objects for the checkers.

In [ ]:
PRICES = {  # USD per 1M tokens
    "gpt-6-astra": {"input": 10.00, "cached": 1.00, "cache_write": 12.50, "output": 50.00},
    "gpt-6.1-sol": {"input": 2.00, "cached": 0.10, "cache_write": 2.50, "output": 10.00},
    "gpt-6-sol": {"input": 2.00, "cached": 0.20, "cache_write": 2.50, "output": 10.00},
    "gpt-6-luna": {"input": 0.10, "cached": 0.01, "cache_write": 0.125, "output": 0.50},
}
SUPPORTED_EFFORT = {
    "gpt-6-astra": {"low", "medium", "high", "xhigh", "max"},
    "gpt-6.1-sol": {"low", "medium", "high", "xhigh", "max"},
    "gpt-6-sol": {"none", "low", "medium", "high", "xhigh", "max"},
    "gpt-6-luna": {"none", "low", "medium", "high", "xhigh", "max"},
}
RESULTS: dict[str, bool] = {}


def mock_response(*, status="completed", reason=None, text=None, refusal=None, function_call=None,
                  reasoning=True, usage=(120, 40, 25), error=None, rid="resp_mock", model="gpt-6-luna"):
    output = []
    if reasoning:
        output.append({"id": "rs_1", "type": "reasoning", "summary": []})
    content = []
    if text is not None:
        content.append({"type": "output_text", "text": text, "annotations": []})
    if refusal is not None:
        content.append({"type": "refusal", "refusal": refusal})
    if content:
        output.append({"id": "msg_1", "type": "message", "role": "assistant",
                       "status": "completed" if status == "completed" else "incomplete", "content": content})
    if function_call:
        output.append({"id": "fc_1", "type": "function_call", "call_id": "call_1",
                       "name": function_call, "arguments": "{}", "status": "completed"})
    inp, out, rsn = usage
    return Response.model_validate({
        "id": rid, "object": "response", "created_at": 0, "model": model, "status": status,
        "incomplete_details": {"reason": reason} if reason else None, "error": error,
        "instructions": None, "metadata": {}, "output": output, "parallel_tool_calls": True,
        "temperature": None, "tool_choice": "auto", "tools": [], "top_p": None,
        "usage": {"input_tokens": inp, "input_tokens_details": {"cached_tokens": 0, "cache_write_tokens": 0},
                  "output_tokens": out, "output_tokens_details": {"reasoning_tokens": rsn},
                  "total_tokens": inp + out},
    })

## Part A — Quiz

Twelve scenario questions. Exactly one answer is correct for each. Write your letters in the next code cell.

**Q1.** Your parser reads `response.output[0].content[0].text`. On `gpt-6-luna` it crashes for many requests even though `status` is `completed`. What is the cause and the right fix?

- A) A content filter removed the text; check `incomplete_details`
- B) The first output item is often a `reasoning` item (or a `function_call`), not a message; use `output_text` or filter `output` items by `type`
- C) Non-streaming responses have no text; set `stream=True`
- D) Text is only returned when `store=True`

**Q2.** You chain turns with `previous_response_id`. Persona rules are passed in `instructions` on the first request only. From turn 2 the tone drifts. What is the best fix?

- A) Set `store=True` explicitly so instructions are saved
- B) Move the rules into the first user message
- C) Raise `reasoning.effort` to `high`
- D) Resend `instructions` on every request

**Q3.** A developer sends both `previous_response_id=prev.id` and `conversation=conv.id` in one `responses.create` call "to be safe". What happens, and what should they do?

- A) The API does not allow both together; pick one state mechanism (for a durable thread, use only `conversation`)
- B) `conversation` silently wins and `previous_response_id` is ignored
- C) It works and the model sees the history twice, which improves recall
- D) It works only with `store=False`

**Q4.** With `max_output_tokens=64` and `reasoning={"effort": "medium"}`, a request returns HTTP 200, `status: "incomplete"`, `incomplete_details.reason: "max_output_tokens"` and `output_text == ""`. What is the most accurate diagnosis?

- A) A transient server error; retry the identical request
- B) The content filter blocked the answer
- C) The budget was used up by reasoning tokens before any visible text; raise `max_output_tokens` or lower the effort
- D) The input exceeded the context window; set `truncation="auto"`

**Q5.** Which statement about data retention is correct?

- A) Conversation objects and their items are deleted after 30 days, like stored responses
- B) Setting `store=False` on a request also deletes the conversation it is attached to
- C) Responses are stored for 30 days by default (`store` defaults to `true`), while Conversation objects and their items have no 30-day TTL and persist until deleted
- D) Neither responses nor conversations are stored unless you opt in

**Q6.** A `gpt-6-astra` analysis at `xhigh` effort can run for several minutes. Mobile clients often disconnect, and the gateway closes idle connections after 60 seconds. Which design fits?

- A) `background=True`, then poll `responses.retrieve` (or receive a `response.completed` webhook), and `responses.cancel` if the user abandons the task
- B) `stream=True` alone, so the connection never looks idle
- C) Raise the client timeout and automatically retry when it times out
- D) Move to Chat Completions, which returns faster

**Q7.** Before routing a request that has two images, twelve function tools and a long system prompt, you need its exact input token count. What should you use?

- A) `tiktoken` on the concatenated text
- B) Character count divided by 4
- C) Send the request and read `usage.input_tokens`, then decide
- D) `client.responses.input_tokens.count(...)` with the same model, instructions, tools and input you will send

**Q8.** A request to `gpt-6-astra` with `reasoning={"effort": "none"}` returns HTTP 400. What is the right change?

- A) `none` requires `stream=True`
- B) GPT-6 Astra does not support `none`; use `low` (or use a model that supports `none`, such as `gpt-6-luna`, if your eval allows it)
- C) The parameter is spelled `reasoning_effort` in the Responses API
- D) Your organization must be verified before using `none`

**Q9.** You migrate a tool-calling support bot to `gpt-6.1-sol` but keep it on Chat Completions. Tool calls stop working. What fixes it?

- A) Move to the Responses API: GPT-6.1 Sol requires Responses for tool calling
- B) Set `reasoning_effort="none"` in Chat Completions
- C) Check `finish_reason == "tool_calls"` instead of `"stop"`
- D) Request `n=2` candidates and choose the one with a tool call

**Q10.** A request to `gpt-6.1-sol` has 300,000 input tokens and produces 2,000 output tokens. How is it priced?

- A) At the normal per-token rates, like a 10K-token prompt
- B) Only the 28,000 tokens above 272K cost double
- C) The whole request is priced at 2x the input/cache rates and 1.5x the output rate
- D) It is rejected unless you set `truncation="auto"`

**Q11.** An intent classifier on `gpt-6-luna` at `low` effort passes an eval example on some runs and fails on others. A teammate wants to add `temperature=0` and `seed=42` "to make CI deterministic". What is the right response?

- A) Do it: together the two parameters guarantee identical outputs
- B) Neither helps here: with reasoning on, GPT-6 models don't support `temperature`, and `responses.create` has no `seed`. Run each example several times, report a pass rate, check properties, and use Structured Outputs for the shape
- C) Add only `top_p=0.1`, which is always supported
- D) Switch to `reasoning={"effort": "max"}`, which makes the output deterministic

**Q12.** A team prototyped a support prompt in the Playground and wants to ship it. Which hand-off matches OpenAI's current guidance?

- A) Save it as a reusable prompt object and pass its `prompt` ID from the app
- B) Copy the prompt into a versioned prompt builder in the repo (typed parameters, passed as `instructions` / `input`), review it in pull requests, and gate changes with an eval set
- C) Ship it unchanged, because it worked in the Playground on the examples they tried
- D) Keep the Playground tab open in production so the prompt can be edited live

In [ ]:
answers = {"q1": None, "q2": None, "q3": None, "q4": None, "q5": None,
           "q6": None, "q7": None, "q8": None, "q9": None, "q10": None,
           "q11": None, "q12": None}

In [ ]:
ANSWER_HASHES = {
    "q1": "96e2e2c94d02b534d4bdc303373df7b704ed4725981fac17549ee985d132c365",
    "q2": "d14b6ae0cbc51f4b715924684653e0d81035a8fe8b79de34ff29eb9db56e2d6b",
    "q3": "2501e2fe43d858f1773d414ee14ab0a8cdf2afefbc4ffd20ae5d956e618f3ea3",
    "q4": "69a9828043db3a177204403c987facd12c855928421b3aeeb5edce8407e3196c",
    "q5": "e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f",
    "q6": "ac78c6455837b3b074080e7912c1bdab72fbbfdcaeefd9269fe293a50c4a9522",
    "q7": "45ff49923feb785985244958470606d87fb75c9912783e470a079e3bb46e1457",
    "q8": "39f95f59afaa9522616af26bd34df1e0184030ae8f738e319fc9a1f15b075834",
    "q9": "8b77cfcdcf7bee660b161787fd482af13fe960d0da61692597e6e870d4668936",
    "q10": "d6ea6e76ddfa07a3b78a8bd000db76a223a8ccc3f38e290cf5a68d95e5494cc5",
    "q11": "42310be80adadfed411c1783b5c42432624ffc39d2ea2f7d837996726d6b0774",
    "q12": "bbd0f1742659cdbbb8582d5974cdc27609e84f189b9c6226758487308eba781e"
}


def grade_quiz(answers: dict) -> int:
    """Compare answers with hashed keys; unanswered counts as wrong."""
    correct = 0
    for qid, expected in ANSWER_HASHES.items():
        letter = str(answers.get(qid) or "").strip().upper()
        ok = bool(letter) and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == expected
        correct += ok
        print(f"{qid:>4}: {letter or '-'} {'✓' if ok else '✗'}")
    print(f"Quiz score: {correct}/{len(ANSWER_HASHES)} ({correct / len(ANSWER_HASHES):.0%})")
    return correct


QUIZ_SCORE = grade_quiz(answers)

## Part B — Coding exercises

Each exercise has a task, a stub to complete, and a checker. Checkers never crash the notebook; they print ✓ or a hint.

### Exercise 1 — Build a stateless request ([1.1](README.md#11-anatomy-of-a-responses-api-request), [1.7](README.md#17-model-selection-and-reasoning-effort))

Write `build_request(question, *, model, instructions, effort="low", max_output_tokens=1500)` that returns the kwargs dict for `client.responses.create`:

- `model`, `instructions`, `max_output_tokens`
- `input`: a list with **one** user message dict (`{"role": "user", "content": question}`)
- `reasoning`: `{"effort": effort}`
- `store=False` (stateless)
- no `temperature` / `top_p`, and no Chat Completions names (`messages`, `max_tokens`)
- raise `ValueError` if `SUPPORTED_EFFORT` lists the model and the effort is not allowed (unknown models pass)

In [ ]:
def build_request(question: str, *, model: str, instructions: str,
                  effort: str = "low", max_output_tokens: int = 1500) -> dict:
    """Return kwargs for client.responses.create (stateless, validated effort)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex1() -> bool:
    try:
        r = build_request("What is a CDN?", model="gpt-6-luna", instructions="Be brief.", effort="none")
        assert isinstance(r, dict), "return a dict of kwargs"
        assert r.get("model") == "gpt-6-luna", "include the model"
        assert r.get("instructions") == "Be brief.", "pass the rules as `instructions`"
        assert r.get("input") == [{"role": "user", "content": "What is a CDN?"}], \
            "input must be a list with ONE user message dict"
        assert r.get("reasoning") == {"effort": "none"}, "reasoning must be {'effort': effort}"
        assert r.get("max_output_tokens") == 1500, "default max_output_tokens is 1500"
        assert r.get("store") is False, "stateless request: store=False"
        assert not {"temperature", "top_p", "max_tokens", "messages"} & r.keys(), \
            "no temperature/top_p, and no Chat Completions names (max_tokens, messages)"
        try:
            build_request("x", model="gpt-6-astra", instructions="i", effort="none")
        except ValueError:
            pass
        else:
            raise AssertionError("gpt-6-astra does not support effort='none': raise ValueError")
        build_request("x", model="some-future-model", instructions="i", effort="high")  # unknown: allowed
        print("✓ Exercise 1 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 1: not implemented yet")
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 1: {type(e).__name__}: {e}")
    return False


RESULTS['ex1'] = check_ex1()

### Exercise 2 — Classify a response ([1.3](README.md#13-status-and-incomplete_details-why-generation-stopped))

Write `classify(resp)` that returns exactly one label:

| Situation | Label |
|---|---|
| `completed` with any `function_call` item (even if there is text too) | `tool_call` |
| `completed` with a `refusal` content part | `refusal` |
| other `completed` | `answer` |
| `incomplete`, reason `max_output_tokens`, some `output_text` | `truncated` |
| `incomplete`, reason `max_output_tokens`, empty `output_text` | `reasoning_exhausted` |
| `incomplete`, reason `content_filter` | `filtered` |
| any other `incomplete` | `other` |
| `failed` | `failed` |
| `queued` or `in_progress` | `pending` |
| `cancelled` | `cancelled` |

In [ ]:
def classify(resp) -> str:
    """Return one label: answer, tool_call, refusal, truncated, reasoning_exhausted,
    filtered, failed, pending, cancelled, other."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex2() -> bool:
    cases = [
        (mock_response(text="Paris."), "answer"),
        (mock_response(text="Checking your order.", function_call="get_order"), "tool_call"),
        (mock_response(refusal="I can't help with that."), "refusal"),
        (mock_response(status="incomplete", reason="max_output_tokens", text="The CAP"), "truncated"),
        (mock_response(status="incomplete", reason="max_output_tokens"), "reasoning_exhausted"),
        (mock_response(status="incomplete", reason="content_filter", text="partial"), "filtered"),
        (mock_response(status="incomplete", reason="steered", text="partial"), "other"),
        (mock_response(status="failed", error={"code": "server_error", "message": "x"}, reasoning=False), "failed"),
        (mock_response(status="queued", reasoning=False), "pending"),
        (mock_response(status="in_progress", reasoning=False), "pending"),
        (mock_response(status="cancelled", reasoning=False), "cancelled"),
    ]
    try:
        for resp, expected in cases:
            got = classify(resp)
            assert got == expected, (f"status={resp.status}, reason="
                                     f"{resp.incomplete_details.reason if resp.incomplete_details else None}, "
                                     f"items={[i.type for i in resp.output]}: expected {expected!r}, got {got!r}")
        print("✓ Exercise 2 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 2: not implemented yet")
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 2: {type(e).__name__}: {e}")
    return False


RESULTS['ex2'] = check_ex2()

### Exercise 3 — Next-turn request for each state pattern ([1.4](README.md#14-conversation-state-stateless-chained-or-durable))

Write `next_request(mode, *, model, instructions, user_text, history=None, last_response_id=None, conversation_id=None)`:

- Always include `model` and `instructions` (they are never carried over).
- `"stateless"`: `store=False`, `input` = a **new** list `history + [user message]` (don't mutate `history`), no server-side state keys.
- `"chain"`: `input=[user message]`; add `previous_response_id` only if `last_response_id` is set; never `conversation`; don't set `store=False`.
- `"conversation"`: `conversation=conversation_id`, `input=[user message]`, never `previous_response_id`; `ValueError` if `conversation_id` is missing.
- Unknown mode: `ValueError`.

In [ ]:
def next_request(mode: str, *, model: str, instructions: str, user_text: str,
                 history: list | None = None, last_response_id: str | None = None,
                 conversation_id: str | None = None) -> dict:
    """Return kwargs for the next turn in "stateless", "chain" or "conversation" mode."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> bool:
    try:
        hist = [{"role": "user", "content": "hi"}, {"role": "assistant", "content": "hello"}]
        hist_copy = list(hist)
        s = next_request("stateless", model="m", instructions="I", user_text="next", history=hist)
        assert s.get("store") is False, "stateless: store=False"
        assert s.get("input") == hist_copy + [{"role": "user", "content": "next"}], "stateless: history + new user msg"
        assert hist == hist_copy, "don't mutate the caller's history list"
        assert "previous_response_id" not in s and "conversation" not in s, "stateless: no server-side state"
        assert s.get("instructions") == "I", "stateless: include instructions on every request"
        c = next_request("chain", model="m", instructions="I", user_text="next", last_response_id="resp_9")
        assert c.get("previous_response_id") == "resp_9", "chain: pass previous_response_id"
        assert c.get("input") == [{"role": "user", "content": "next"}], "chain: send only the new message"
        assert c.get("instructions") == "I", "chain: instructions are NOT carried over, resend them"
        assert "conversation" not in c and c.get("store") is not False, "chain: no conversation, don't disable store"
        first = next_request("chain", model="m", instructions="I", user_text="hi")
        assert "previous_response_id" not in first, "chain, first turn: no previous_response_id yet"
        v = next_request("conversation", model="m", instructions="I", user_text="next", conversation_id="conv_1")
        assert v.get("conversation") == "conv_1" and "previous_response_id" not in v, \
            "conversation: use conversation only (never together with previous_response_id)"
        assert v.get("instructions") == "I", "conversation: resend instructions too"
        assert v.get("input") == [{"role": "user", "content": "next"}], "conversation: input = [the new user message]"
        for mode, req in (("stateless", s), ("chain", c), ("chain, first turn", first), ("conversation", v)):
            assert req.get("model") == "m", f"{mode}: include model on every request"
        for bad in (lambda: next_request("conversation", model="m", instructions="I", user_text="x"),
                    lambda: next_request("magic", model="m", instructions="I", user_text="x")):
            try:
                bad()
            except ValueError:
                pass
            else:
                raise AssertionError("raise ValueError for an unknown mode or a missing conversation_id")
        print("✓ Exercise 3 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 3: not implemented yet")
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 3: {type(e).__name__}: {e}")
    return False


RESULTS['ex3'] = check_ex3()

### Exercise 4 — Cost from `usage` ([1.6](README.md#16-context-window-tokens-and-cost))

Write `cost_usd(usage, model)` using `PRICES` (USD per million tokens):

- uncached input = `input_tokens - cached_tokens - cache_write_tokens`, each billed at its own rate
- output at the output rate; **reasoning tokens are already inside `output_tokens`**
- if `input_tokens > 272_000`: multiply all input-side costs by 2 and output by 1.5
- `ValueError` for a model missing from `PRICES`

In [ ]:
def cost_usd(usage, model: str) -> float:
    """Estimate USD cost of one response from its usage object and PRICES."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> bool:
    def u(inp, out, cached=0, writes=0, reasoning=0):
        return SimpleNamespace(input_tokens=inp, output_tokens=out,
                               input_tokens_details=SimpleNamespace(cached_tokens=cached, cache_write_tokens=writes),
                               output_tokens_details=SimpleNamespace(reasoning_tokens=reasoning))
    try:
        close = lambda a, b: abs(a - b) < 1e-9  # noqa: E731
        got = cost_usd(u(10_000, 1_000, cached=4_000), "gpt-6-luna")
        assert close(got, 0.00114), f"luna with 4K cached tokens: expected 0.00114, got {got} (cached tokens use the cached rate)"
        got = cost_usd(u(5_000, 500, cached=1_000, writes=2_000), "gpt-6-astra")
        assert close(got, 0.071), f"astra with cache writes: expected 0.071, got {got}"
        got = cost_usd(u(300_000, 2_000), "gpt-6.1-sol")
        assert close(got, 1.23), f"300K-token prompt: expected 1.23 (2x input, 1.5x output), got {got}"
        a = cost_usd(u(1_000, 1_000, reasoning=800), "gpt-6-luna")
        b = cost_usd(u(1_000, 1_000, reasoning=0), "gpt-6-luna")
        assert close(a, b), "reasoning tokens are already part of output_tokens: don't count them twice"
        try:
            cost_usd(u(1, 1), "unknown-model")
        except ValueError:
            pass
        else:
            raise AssertionError("raise ValueError for a model with no prices")
        print("✓ Exercise 4 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 4: not implemented yet")
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 4: {type(e).__name__}: {e}")
    return False


RESULTS['ex4'] = check_ex4()

### Exercise 5 — Accumulate a stream ([1.5](README.md#15-streaming-background-mode-and-webhooks))

Write `collect_stream(events)` that walks typed stream events and returns `{"text", "status", "cursor", "total_tokens"}`:

- join every `response.output_text.delta` event's `delta`
- `cursor` = the last `sequence_number` seen (`None` if no events)
- on `response.completed` / `response.incomplete` / `response.failed`, read `event.response.status` and `event.response.usage.total_tokens`
- ignore other event types; if no terminal event arrives, `status` and `total_tokens` stay `None`

In [ ]:
def collect_stream(events) -> dict:
    """Return {"text": str, "status": str | None, "cursor": int | None, "total_tokens": int | None}."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> bool:
    ev = lambda **kw: SimpleNamespace(**kw)  # noqa: E731
    done = ev(type="response.completed", sequence_number=5,
              response=ev(status="completed", usage=ev(total_tokens=57)))
    events = [ev(type="response.created", sequence_number=0),
              ev(type="response.output_item.added", sequence_number=1),
              ev(type="response.output_text.delta", delta="Safe ", sequence_number=2),
              ev(type="response.output_text.delta", delta="to ", sequence_number=3),
              ev(type="response.output_text.delta", delta="retry.", sequence_number=4), done]
    try:
        r = collect_stream(events)
        assert r == {"text": "Safe to retry.", "status": "completed", "cursor": 5, "total_tokens": 57}, f"got {r}"
        cut = collect_stream(events[:4])
        assert cut == {"text": "Safe to ", "status": None, "cursor": 3, "total_tokens": None}, \
            f"stream without a terminal event: got {cut}"
        inc = collect_stream([ev(type="response.output_text.delta", delta="Part", sequence_number=0),
                              ev(type="response.incomplete", sequence_number=1,
                                 response=ev(status="incomplete", usage=ev(total_tokens=99)))])
        assert inc["status"] == "incomplete" and inc["total_tokens"] == 99, "handle response.incomplete too"
        print("✓ Exercise 5 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 5: not implemented yet")
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 5: {type(e).__name__}: {e}")
    return False


RESULTS['ex5'] = check_ex5()

### Exercise 6 — Remember across turns (live — calls the API, ~$0.001, optional, not scored) ([1.4](README.md#14-conversation-state-stateless-chained-or-durable))

Write `remember_across_turns(client, model)`: turn 1 says "Hi, my name is Lena."; turn 2 chains with `previous_response_id` (resend your `instructions`) and asks "What is my name?". Return `(r1, r2)`. Use `reasoning={"effort": "low"}` and a small `max_output_tokens`.

In [ ]:
def remember_across_turns(client, model: str):
    """(live) Turn 1: tell the model your name is Lena. Turn 2: chain with previous_response_id
    (resend instructions!) and ask "What is my name?". Return (r1, r2)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex6() -> bool:
    try:
        r1, r2 = remember_across_turns(client, MODEL)
        assert r1.status == "completed" and r2.status == "completed", f"statuses: {r1.status}, {r2.status}"
        assert r2.previous_response_id == r1.id, "turn 2 must chain from turn 1 with previous_response_id"
        assert "lena" in r2.output_text.lower(), f"the model should remember the name: {r2.output_text!r}"
        print("✓ Exercise 6 (live) passed")
        for r in (r1, r2):
            client.responses.delete(r.id)   # clean up stored responses
        return True
    except NotImplementedError:
        print("✗ Exercise 6: not implemented yet (optional)")
    except NameError:
        print("✗ Exercise 6: run the setup cell that creates `client` first (needs OPENAI_API_KEY)")
    except openai.APIError as e:
        print(f"✗ Exercise 6: API error ({type(e).__name__}); check your key and model")
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 6: {type(e).__name__}: {e}")
    return False


LIVE_OK = check_ex6()

## Part C — Architecture scenario

**Scenario.** An EU fintech wants a customer-support assistant on the OpenAI platform. Requirements:

- The organization has **Zero Data Retention** and EU data residency.
- Conversations last up to ~150 turns over a week; the assistant must remember exact figures (amounts, dates, case IDs) the customer gave.
- About 5% of requests trigger a "deep analysis" of the customer's transaction history that can take several minutes.
- Monthly budget is tight; most turns are simple FAQ answers.

**Your task.** Describe: (1) how conversation state is stored and carried between turns, (2) how you keep long conversations within cost and context limits without losing exact figures, (3) how the deep-analysis requests run, (4) model and effort choices and how you'd justify them, (5) how your code decides whether a response is usable.

_Write your answer in the markdown cell below._

**My answer:**

1. State:
2. Long conversations:
3. Deep analysis:
4. Model and effort:
5. Response handling:

<details><summary>Rubric: what a strong answer contains</summary>

- **State:** ZDR means `store` is always treated as `false`, and the Conversations API is not ZDR-eligible, so state lives in the app: manual history, replaying **every** output item (reasoning items carry `encrypted_content` when `store=False`, which preserves reasoning continuity). Not `previous_response_id` over HTTP: OpenAI pairs it with stored responses, and with `store=False` nothing is persisted to load (only WebSocket mode keeps an in-memory copy, until it is evicted). Instructions are sent on every request from versioned code (not `v1/prompts`).
- **Long conversations:** a structured "case facts" block (amounts, dates, case IDs) kept verbatim and never summarized; older chit-chat compacted with server-side compaction (`context_management` with `compact_threshold`) or the standalone `responses.compact` endpoint, both ZDR-friendly with `store=False`; stable prefix first so automatic prompt caching applies; `input_tokens.count` before large requests; a guard far below the 272K long-context price threshold; not `truncation="auto"` (it drops the oldest items, which may hold the facts).
- **Deep analysis:** `background=True` with polling (or a `response.completed` webhook) and `cancel` on abandonment; aware that with `store=False`/ZDR the background data is kept only ~10 minutes for polling, so results must be collected promptly; optional `stream=True` with a `sequence_number` cursor for progress.
- **Model and effort:** an eval set from real tickets; Luna at `low` (or `none`) for FAQ turns, stepping up (6.1 Sol or Astra, higher effort) only for deep analysis if evals demand it; decisions measured as cost per successful task; data residency eligibility checked per model and processing mode (for example, Fast mode is not available with EU data residency for Astra, Sol and Luna).
- **Response handling:** branch on `status` → `incomplete_details.reason` → item types; retry `max_output_tokens` truncation with a bigger budget or lower effort; treat `content_filter` and refusals explicitly; typed SDK exceptions for HTTP errors, respecting `Retry-After`; log usage (including reasoning and cached tokens) per turn for cost tracking.
- **Red flags:** relying on the Conversations API or `previous_response_id` under ZDR; "use the 1M window, no summarization needed"; reading `output[0].content[0].text`; long synchronous requests with client-side retries.
</details>

## Scorecard

In [ ]:
quiz = globals().get("QUIZ_SCORE", 0)
passed = sum(bool(globals().get("RESULTS", {}).get(ex)) for ex in ["ex1", "ex2", "ex3", "ex4", "ex5"])
overall = (quiz + passed) / 17
print(f"Quiz: {quiz}/12 ({quiz / 12:.0%})")
print(f"Exercises passed: {passed}/5")
print(f"Live exercise (bonus): {'passed' if globals().get('LIVE_OK') else 'not passed / skipped'}")
print(f"Overall: {overall:.0%} -> {'PASS' if overall >= 0.72 else 'RETRY'} (pass mark 72%)")

**Next chapter:** [Chapter 2 — Function Calling and Structured Outputs](../02-function-calling-and-structured-outputs/README.md).

---

## Solutions (spoilers)

### Quiz answers

<details><summary>Q1</summary>

**B.** The `output` array often starts with a `reasoning` item, and tool calls appear as `function_call` items. OpenAI's docs warn that text is not guaranteed at `output[0].content[0].text`. A would show `status: incomplete`; C and D are false.
</details>

<details><summary>Q2</summary>

**D.** `instructions` apply to the current request only; earlier instructions are not carried over by `previous_response_id`. A changes nothing (store is already the default), B lowers the authority of the rules, C does not restore missing instructions.
</details>

<details><summary>Q3</summary>

**A.** The API reference states that `previous_response_id` cannot be used in conjunction with `conversation`. B, C and D describe behaviors the API does not have.
</details>

<details><summary>Q4</summary>

**C.** `max_output_tokens` covers reasoning plus visible output. An empty `output_text` with this reason means reasoning consumed the whole budget, and you were still billed. A repeats the failure, B would report `content_filter`, D would be a 400 error with the default `truncation`.
</details>

<details><summary>Q5</summary>

**C.** The conversation-state guide says responses are saved for 30 days by default and that conversation objects and items are not subject to the 30-day TTL. A and D are false, and B invents a behavior.
</details>

<details><summary>Q6</summary>

**A.** Background mode runs the job server-side and is designed for long-running reasoning; polling or webhooks survive disconnects. B still breaks when the client drops (unless combined with background mode), C re-runs and re-bills the job, D does not solve the transport problem.
</details>

<details><summary>Q7</summary>

**D.** The counting endpoint accepts the same input format as `responses.create` (instructions, input items including images, tools, conversation state) and includes formatting tokens. Generation-only options such as `max_output_tokens` or `stream` are not part of the count request. A and B can't count images, tools or formatting tokens. C spends money before you have the answer.
</details>

<details><summary>Q8</summary>

**B.** Supported effort values are model-specific: Astra and 6.1 Sol do not accept `none`; Luna and GPT-6 Sol do. C is the Chat Completions spelling, and A and D are invented.
</details>

<details><summary>Q9</summary>

**A.** GPT-6.1 Sol supports Chat Completions only without tool calling. B does not work because 6.1 Sol does not support `none` (that workaround applies to GPT-6 Sol and Luna). C and D don't make tool calling available.
</details>

<details><summary>Q10</summary>

**C.** The model pages state that prompts with more than 272K input tokens are priced at 2x input and cache rates and 1.5x output for the full request. Inputs up to 922K are accepted.
</details>

<details><summary>Q11</summary>

**B.** OpenAI says to remove `temperature`, `top_p` and `top_logprobs` from GPT-6 requests whenever reasoning effort is not `none`, and the Responses API has no `seed` parameter. Output is non-deterministic, so evals measure a pass rate over several runs. C is wrong for the same reason as A, and D only changes how much the model reasons.
</details>

<details><summary>Q12</summary>

**B.** Reusable prompt objects (`v1/prompts`) are deprecated and shut down on November 30, 2026; OpenAI's prompting guide says to treat prompts as application code with tests and evals. C confuses a few manual tries with evidence, and D is not a deployment model.
</details>

<details><summary>Exercise 1 — reference solution</summary>

```python
def build_request(question: str, *, model: str, instructions: str,
                  effort: str = "low", max_output_tokens: int = 1500) -> dict:
    allowed = SUPPORTED_EFFORT.get(model)
    if allowed is not None and effort not in allowed:
        raise ValueError(f"{model} does not support effort={effort!r}")
    return {
        "model": model,
        "instructions": instructions,
        "input": [{"role": "user", "content": question}],
        "reasoning": {"effort": effort},
        "max_output_tokens": max_output_tokens,
        "store": False,
    }
```
</details>

<details><summary>Exercise 2 — reference solution</summary>

```python
def classify(resp) -> str:
    if resp.status == "completed":
        if any(item.type == "function_call" for item in resp.output):
            return "tool_call"
        if any(part.type == "refusal" for item in resp.output if item.type == "message"
               for part in item.content):
            return "refusal"
        return "answer"
    if resp.status == "incomplete":
        reason = resp.incomplete_details.reason if resp.incomplete_details else None
        if reason == "max_output_tokens":
            return "truncated" if resp.output_text else "reasoning_exhausted"
        if reason == "content_filter":
            return "filtered"
        return "other"
    if resp.status == "failed":
        return "failed"
    if resp.status in ("queued", "in_progress"):
        return "pending"
    if resp.status == "cancelled":
        return "cancelled"
    return "other"
```
</details>

<details><summary>Exercise 3 — reference solution</summary>

```python
def next_request(mode: str, *, model: str, instructions: str, user_text: str,
                 history: list | None = None, last_response_id: str | None = None,
                 conversation_id: str | None = None) -> dict:
    user_msg = {"role": "user", "content": user_text}
    kwargs = {"model": model, "instructions": instructions}   # instructions resent every turn
    if mode == "stateless":
        kwargs.update(store=False, input=list(history or []) + [user_msg])
    elif mode == "chain":
        kwargs["input"] = [user_msg]
        if last_response_id:
            kwargs["previous_response_id"] = last_response_id
    elif mode == "conversation":
        if not conversation_id:
            raise ValueError("conversation mode needs a conversation_id")
        kwargs.update(conversation=conversation_id, input=[user_msg])
    else:
        raise ValueError(f"unknown mode {mode!r}")
    return kwargs
```
</details>

<details><summary>Exercise 4 — reference solution</summary>

```python
def cost_usd(usage, model: str) -> float:
    if model not in PRICES:
        raise ValueError(f"no prices for {model}")
    p = PRICES[model]
    cached = usage.input_tokens_details.cached_tokens
    writes = usage.input_tokens_details.cache_write_tokens
    uncached = usage.input_tokens - cached - writes
    in_mult, out_mult = (2.0, 1.5) if usage.input_tokens > 272_000 else (1.0, 1.0)
    input_cost = (uncached * p["input"] + cached * p["cached"] + writes * p["cache_write"]) * in_mult
    # reasoning tokens are already included in output_tokens: don't add them again
    return (input_cost + usage.output_tokens * p["output"] * out_mult) / 1e6
```
</details>

<details><summary>Exercise 5 — reference solution</summary>

```python
def collect_stream(events) -> dict:
    parts, status, cursor, total = [], None, None, None
    for ev in events:
        cursor = getattr(ev, "sequence_number", cursor)
        if ev.type == "response.output_text.delta":
            parts.append(ev.delta)
        elif ev.type in ("response.completed", "response.incomplete", "response.failed"):
            status = ev.response.status
            usage = getattr(ev.response, "usage", None)
            total = usage.total_tokens if usage else None
    return {"text": "".join(parts), "status": status, "cursor": cursor, "total_tokens": total}
```
</details>

<details><summary>Exercise 6 (live) — reference solution</summary>

```python
def remember_across_turns(client, model: str):
    rules = "Answer in one short sentence."
    r1 = client.responses.create(model=model, instructions=rules, input="Hi, my name is Lena.",
                                 reasoning={"effort": "low"}, max_output_tokens=400)
    r2 = client.responses.create(model=model, instructions=rules, previous_response_id=r1.id,
                                 input="What is my name?", reasoning={"effort": "low"}, max_output_tokens=400)
    return r1, r2
```
</details>